# Cost & Token Budgeting, and Logging
**Module 5, Topic 8 — Automation & Agentic Systems**

## The Problem

An LLM agent that loops — triage, search, draft, review — makes **multiple API calls per question**. Without guardrails, a prompt injection that triggers an infinite loop or a badly worded question that causes excessive retries can burn through your token budget in minutes. And without logging, you won't know it happened until the invoice arrives.

## The Solution

**Measure, budget, cache, and log.** Read `response.usage` on every call (ground truth, not estimates). Set per-request and daily token budgets that stop runaway agents. Cache identical requests so you never pay twice for the same answer. Log every call with sanitised previews so you can audit cost spikes.

In this notebook you will:
1. Measure token use from `response.usage`
2. Build a **per-request token budget** that stops runaway loops
3. Build a **daily budget** with persistent tracking
4. Add **hash-based caching** for repeated calls
5. Add **sanitised JSON logging** (redact sensitive data)
6. Wire it all into a **complete pipeline** and test with NOC alerts

**Time:** 35 minutes | **Difficulty:** ⭐⭐⭐☆☆

## Setup

In [ ]:
import os
import json
import re
import time
import uuid
import hashlib
from datetime import date, datetime
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")

print(f"Client ready — model: {MODEL}")

## Step 1 — Measuring Token Use

Every API response includes a `usage` object with exact token counts. This is the ground truth for cost — never estimate from character counts.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a NOC alert classifier. Classify alerts as: critical, warning, or info."},
        {"role": "user", "content": "Transponder B3 on NigComSat-1R shows 2 dB signal degradation over 48 hours."},
    ],
)

usage = response.usage
print(f"Input tokens:  {usage.prompt_tokens}")
print(f"Output tokens: {usage.completion_tokens}")
print(f"Total tokens:  {usage.total_tokens}")
print(f"\nClassification: {response.choices[0].message.content}")

### Multiple calls add up

In [ ]:
alerts = [
    "Transponder B3 on NigComSat-1R shows 2 dB signal degradation over 48 hours.",
    "Abuja Earth Station backup generator test completed — all systems nominal.",
    "Ka-band uplink from Lagos hub lost for 35 seconds at 02:14 UTC, auto-recovered.",
]

total_tokens = 0
for i, alert in enumerate(alerts, 1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Classify this NOC alert as: critical, warning, or info. Reply with the category only."},
            {"role": "user", "content": alert},
        ],
    )
    tokens = resp.usage.total_tokens
    total_tokens += tokens
    print(f"Alert {i}: {tokens} tokens — {resp.choices[0].message.content}")

print(f"\nTotal across 3 calls: {total_tokens} tokens")
print(f"If this ran daily for 30 days: ~{total_tokens * 30:,} tokens/month")

## Step 2 — Per-Request Token Budget

A `TokenBudget` stops the agent if a single request burns too many tokens. This catches runaway loops and prompt injection attacks.

In [ ]:
class TokenBudget:
    def __init__(self, max_tokens_per_request=10000):
        self.max_per_request = max_tokens_per_request
        self.used = 0

    def track(self, usage):
        self.used += usage.total_tokens
        if self.used > self.max_per_request:
            raise RuntimeError(
                f"Token budget exceeded: {self.used} > {self.max_per_request}"
            )
        return self.used

    def reset(self):
        self.used = 0

    def remaining(self):
        return max(0, self.max_per_request - self.used)


print("TokenBudget class defined")

### Budget within limits

In [ ]:
budget = TokenBudget(max_tokens_per_request=5000)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Classify this NOC alert as: critical, warning, or info. Reply with the category only."},
        {"role": "user", "content": "Transponder B3 on NigComSat-1R shows 2 dB signal degradation over 48 hours."},
    ],
)

used = budget.track(response.usage)
print(f"Used so far: {used} tokens")
print(f"Remaining:   {budget.remaining()} tokens")
print(f"Classification: {response.choices[0].message.content}")

### Budget exceeded — simulating a runaway loop

In [ ]:
tight_budget = TokenBudget(max_tokens_per_request=150)

alerts_to_classify = [
    "Transponder B3 signal degradation.",
    "Backup generator test passed.",
    "Ka-band uplink lost for 35 seconds.",
]

for i, alert in enumerate(alerts_to_classify, 1):
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Classify as: critical, warning, or info."},
                {"role": "user", "content": alert},
            ],
        )
        used = tight_budget.track(resp.usage)
        print(f"Alert {i}: {resp.usage.total_tokens} tokens (total: {used}) — {resp.choices[0].message.content}")
    except RuntimeError as e:
        print(f"Alert {i}: STOPPED — {e}")
        break

## Step 3 — Daily Budget

Caps total spending across all requests in a day. Uses a JSON file so the count persists between runs.

In [ ]:
BUDGET_FILE = "daily_budget.json"


def load_daily_usage():
    today = str(date.today())
    if os.path.exists(BUDGET_FILE):
        with open(BUDGET_FILE) as f:
            data = json.load(f)
        return data.get(today, 0)
    return 0


def save_daily_usage(tokens):
    today = str(date.today())
    data = {}
    if os.path.exists(BUDGET_FILE):
        with open(BUDGET_FILE) as f:
            data = json.load(f)
    data[today] = tokens
    with open(BUDGET_FILE, "w") as f:
        json.dump(data, f, indent=2)


def check_daily_budget(tokens_used, daily_limit=50000):
    current = load_daily_usage()
    new_total = current + tokens_used
    if new_total > daily_limit:
        raise RuntimeError(
            f"Daily budget exceeded: {new_total:,} > {daily_limit:,} tokens"
        )
    save_daily_usage(new_total)
    pct = (new_total / daily_limit) * 100
    return new_total, pct


print("Daily budget functions defined")

In [ ]:
# Reset the budget file for a clean demo
if os.path.exists(BUDGET_FILE):
    os.remove(BUDGET_FILE)

for i, tokens in enumerate([120, 95, 140], 1):
    total, pct = check_daily_budget(tokens, daily_limit=500)
    print(f"Request {i}: +{tokens} tokens -> daily total: {total} ({pct:.0f}%)")

try:
    check_daily_budget(200, daily_limit=500)
    print("Request 4: passed")
except RuntimeError as e:
    print(f"Request 4: STOPPED — {e}")

if os.path.exists(BUDGET_FILE):
    os.remove(BUDGET_FILE)

## Step 4 — Caching Repeated Calls

If the same messages are sent twice, the cached response is returned — no API call, no tokens, no cost.

In [ ]:
_cache = {}


def cached_completion(messages, model=MODEL):
    key = hashlib.sha256(
        json.dumps(messages, sort_keys=True).encode()
    ).hexdigest()

    if key in _cache:
        print("  [cache hit — 0 tokens]")
        return _cache[key]

    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )
    print(f"  [cache miss — {response.usage.total_tokens} tokens]")
    _cache[key] = response
    return response


print("cached_completion defined")

In [ ]:
messages = [
    {"role": "system", "content": "Classify as: critical, warning, or info."},
    {"role": "user", "content": "Transponder B3 signal degradation over 48 hours."},
]

print("Call 1:")
r1 = cached_completion(messages)
print(f"  Result: {r1.choices[0].message.content}")

print("\nCall 2 (same messages):")
r2 = cached_completion(messages)
print(f"  Result: {r2.choices[0].message.content}")

print("\nCall 3 (different message):")
r3 = cached_completion([
    {"role": "system", "content": "Classify as: critical, warning, or info."},
    {"role": "user", "content": "Backup generator test passed — all nominal."},
])
print(f"  Result: {r3.choices[0].message.content}")

print(f"\nCache size: {len(_cache)} entries")

## Step 5 — Sanitised JSON Logging

Log tokens, cost, latency, and model per request. Sanitise sensitive data before writing.

In [ ]:
def sanitise_for_log(text, max_length=200):
    """Truncate and redact before logging."""
    truncated = text[:max_length] + ("..." if len(text) > max_length else "")
    return re.sub(r"VSAT-OP-\d{4}", "VSAT-OP-XXXX", truncated)


print(sanitise_for_log("Alert for licence VSAT-OP-0192 at Abuja station"))
print(sanitise_for_log("A" * 300))

In [ ]:
def logged_completion(messages, model=MODEL):
    """Make an API call and log the cost data."""
    request_id = str(uuid.uuid4())[:8]
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    elapsed_ms = int((time.time() - start) * 1000)
    usage = response.usage

    log_entry = {
        "request_id": request_id,
        "timestamp": datetime.now().isoformat(),
        "model": model,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_ms": elapsed_ms,
        "status": "success",
        "input_preview": sanitise_for_log(messages[-1]["content"]),
    }

    print(json.dumps(log_entry, indent=2))
    return response, log_entry


print("logged_completion defined")

In [ ]:
response, log = logged_completion([
    {"role": "system", "content": "Classify as: critical, warning, or info."},
    {"role": "user", "content": "Ka-band uplink from Lagos hub lost for 35 seconds at 02:14 UTC, auto-recovered. Licence VSAT-OP-0192."},
])

print(f"\nClassification: {response.choices[0].message.content}")
print(f"Licence number redacted in log: {'VSAT-OP-XXXX' in log['input_preview']}")

## Step 6 — Complete Pipeline: Budget + Cache + Logging

Wire everything together: per-request budget, daily budget, caching, and logging.

In [ ]:
_cache.clear()
if os.path.exists(BUDGET_FILE):
    os.remove(BUDGET_FILE)


def budgeted_completion(messages, model=MODEL,
                        request_budget=5000, daily_limit=50000):
    """API call with budget, cache, and logging."""
    cache_key = hashlib.sha256(
        json.dumps(messages, sort_keys=True).encode()
    ).hexdigest()

    if cache_key in _cache:
        print("  [cache hit — 0 tokens, 0 cost]")
        return _cache[cache_key], None

    request_id = str(uuid.uuid4())[:8]
    start = time.time()

    response = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    elapsed_ms = int((time.time() - start) * 1000)
    usage = response.usage

    if usage.total_tokens > request_budget:
        raise RuntimeError(
            f"Per-request budget exceeded: {usage.total_tokens} > {request_budget}"
        )

    daily_total, pct = check_daily_budget(usage.total_tokens, daily_limit)

    log_entry = {
        "request_id": request_id,
        "timestamp": datetime.now().isoformat(),
        "model": model,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
        "latency_ms": elapsed_ms,
        "daily_total": daily_total,
        "daily_pct": f"{pct:.0f}%",
        "status": "success",
    }

    _cache[cache_key] = response

    return response, log_entry


print("budgeted_completion defined")

### Run the pipeline on a batch of NOC alerts

In [ ]:
noc_alerts = [
    "Transponder B3 on NigComSat-1R shows 2 dB signal degradation over 48 hours.",
    "Abuja Earth Station backup generator test completed — all systems nominal.",
    "Ka-band uplink from Lagos hub lost for 35 seconds at 02:14 UTC, auto-recovered.",
    "Transponder B3 on NigComSat-1R shows 2 dB signal degradation over 48 hours.",  # duplicate
    "Port Harcourt VSAT terminal VSAT-OP-0341 reports intermittent connectivity.",
]

print("Processing 5 NOC alerts (one is a duplicate):\n")
for i, alert in enumerate(noc_alerts, 1):
    messages = [
        {"role": "system", "content": "Classify this NOC alert as: critical, warning, or info. Reply with the category only."},
        {"role": "user", "content": alert},
    ]
    try:
        resp, log = budgeted_completion(messages, daily_limit=1000)
        result = resp.choices[0].message.content
        if log:
            print(f"  Alert {i}: {result} — {log['total_tokens']} tokens (daily: {log['daily_pct']})")
        else:
            print(f"  Alert {i}: {result}")
    except RuntimeError as e:
        print(f"  Alert {i}: STOPPED — {e}")
        break

if os.path.exists(BUDGET_FILE):
    os.remove(BUDGET_FILE)

Alert 4 is a duplicate of Alert 1 — the cache returned it instantly at zero cost.

## Step 7 — Cost Alerting

Warn when the daily budget is approaching its limit.

In [ ]:
def check_daily_budget_with_alert(tokens_used, daily_limit=50000, warn_pct=80):
    """Check daily budget and emit alerts at thresholds."""
    current = load_daily_usage()
    new_total = current + tokens_used

    if new_total > daily_limit:
        alert = {
            "level": "CRITICAL",
            "message": f"Daily budget EXCEEDED: {new_total:,} / {daily_limit:,} tokens",
            "timestamp": datetime.now().isoformat(),
        }
        print(f"COST ALERT: {json.dumps(alert)}")
        raise RuntimeError(alert["message"])

    save_daily_usage(new_total)
    pct = (new_total / daily_limit) * 100

    if pct >= warn_pct:
        alert = {
            "level": "WARNING",
            "message": f"Daily budget {pct:.0f}% consumed: {new_total:,} / {daily_limit:,} tokens",
            "timestamp": datetime.now().isoformat(),
        }
        print(f"COST WARNING: {alert['message']}")

    return new_total, pct


if os.path.exists(BUDGET_FILE):
    os.remove(BUDGET_FILE)

print("Simulating requests against a 500-token daily budget:\n")
for i, tokens in enumerate([150, 120, 100, 80, 60], 1):
    try:
        total, pct = check_daily_budget_with_alert(tokens, daily_limit=500, warn_pct=80)
        print(f"  Request {i}: +{tokens} -> {total} tokens ({pct:.0f}%)")
    except RuntimeError as e:
        print(f"  Request {i}: {e}")
        break

if os.path.exists(BUDGET_FILE):
    os.remove(BUDGET_FILE)

## Step 8 — NOC Digest Budget Calculation

Estimate the cost of the daily digest automation and set appropriate limits.

In [ ]:
sample_alerts = [
    "02:14 — Ka-band uplink from Lagos hub lost for 35 seconds, auto-recovered.",
    "04:30 — Transponder B3 signal degradation: 2 dB over 48 hours.",
    "06:00 — Abuja Earth Station backup generator test — all nominal.",
    "09:15 — Port Harcourt VSAT terminal VSAT-OP-0341 intermittent connectivity.",
    "11:42 — Kano hub bandwidth utilisation at 92% — approaching capacity.",
]

digest_prompt = [
    {"role": "system", "content": (
        "You are a NOC shift supervisor assistant. Summarise the alerts below into a morning digest. "
        "Group by severity (critical, warning, info). Highlight items requiring follow-up."
    )},
    {"role": "user", "content": "\n".join(sample_alerts)},
]

resp = client.chat.completions.create(model=MODEL, messages=digest_prompt)
usage = resp.usage

print("=== Digest ===")
print(resp.choices[0].message.content)
print("\n=== Cost Estimate ===")
print(f"This run: {usage.total_tokens} tokens ({usage.prompt_tokens} in, {usage.completion_tokens} out)")
print(f"Daily (1 run):   {usage.total_tokens} tokens")
print(f"Monthly (30 runs): ~{usage.total_tokens * 30:,} tokens")
print(f"\nRecommended per-request limit: {usage.total_tokens * 3:,} tokens (3x expected)")
print(f"Recommended daily limit:       {usage.total_tokens * 4:,} tokens (room for retries)")

## Summary

- **`response.usage`** gives exact token counts — never estimate from character counts
- **Per-request budget** stops runaway agents and injection-caused loops
- **Daily budget** caps total daily spending across all requests
- **Caching** (hash-based) avoids paying twice for identical requests
- **Logging** records tokens, cost, latency per request with sanitised previews
- **Alerting** at 80% and 100% thresholds catches cost spikes before the invoice
- **Cost control is also security** — an injection that causes looping triggers a cost alert

## Exercises

### Exercise 1 — TTL-based cache expiry
Modify `cached_completion` to store the timestamp with each cache entry. Add a `ttl_seconds` parameter (default 300). On a cache hit, check whether the entry is older than the TTL — if so, treat it as a miss and make a fresh call. Test by setting `ttl_seconds=5`, making a call, waiting 6 seconds, and confirming the second call is a miss.

### Exercise 2 — Log to a file
Modify `logged_completion` to append each `log_entry` as a JSON line to `api_calls.jsonl` (one JSON object per line). After running 3 calls, read the file back and print the total tokens across all entries. Clean up the file at the end.

### Exercise 3 — Budget the compliance-checker agent
Import the compliance-checker graph from Topic 3 (or copy the relevant cells). Wrap every `client.chat.completions.create` call inside `budgeted_completion`. Set a per-request limit of 2000 tokens and a daily limit of 5000. Run the four test questions from Topic 3 and check whether any of them exceed the per-request budget.

**Time:** 25 minutes

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here
